# Detector 2 Training — Adaptive Road Vision

Second object-detector experiment using the same final dataset and evaluation protocol as Detector 1.

**Model:** Ultralytics YOLO11s (`yolo11s.pt`)  
**Dataset:** automatically discovers `final_condition_balanced_v2`, `final_condition_balanced`, or `final` under `datasets/`  
**Purpose:** train a second detector with a larger YOLO backbone for a fair comparison with Detector 1.

**Important:** verify that `configs/detection.yaml` uses the exact same class order as Detector 1. The test split is reserved for final evaluation only.


In [ ]:
from pathlib import Path
import os, json, shutil, random, sys
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path(r'C:\Users\ankit\Desktop\Adaptive_Road_Vision')
if not PROJECT_ROOT.exists():
    # Useful fallback when the notebook is run from inside the project directory.
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    PROJECT_ROOT = next((p for p in candidates if (p / 'datasets').exists() and (p / 'models').exists()), PROJECT_ROOT)
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {PROJECT_ROOT}. Update PROJECT_ROOT to your local Adaptive_Road_Vision folder.'
    )
os.chdir(PROJECT_ROOT)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Project root:', PROJECT_ROOT)
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Install/check Ultralytics if needed.
try:
    import ultralytics
    from ultralytics import YOLO
    print('Ultralytics:', ultralytics.__version__)
except ImportError:
    print('Ultralytics is not installed in this environment.')
    print('Run: pip install ultralytics pyyaml')
    raise


## 1. Experiment configuration

Detector 2 keeps the dataset split and evaluation protocol unchanged. The main architecture change from Detector 1 is `yolo11s.pt` instead of `yolo11n.pt`.

In [ ]:
MODEL_NAME = 'yolo11s.pt'
EXPERIMENT_NAME = 'detector_2'
IMAGE_SIZE = 640
BATCH_SIZE = 16
EPOCHS = 50
PATIENCE = 10
WORKERS = 0  # Windows-safe
SEED = 42


def resolve_dataset_root():
    candidates = [
        PROJECT_ROOT / 'datasets' / 'final_condition_balanced_v2',
        PROJECT_ROOT / 'datasets' / 'final_condition_balanced',
        PROJECT_ROOT / 'datasets' / 'final',
    ]
    for cand in candidates:
        has_splits = all((cand / 'images' / s).is_dir() and (cand / 'labels' / s).is_dir()
                         for s in ('train', 'val', 'test'))
        has_metadata = (cand / 'metadata' / 'final_dataset_metadata.csv').exists() or (cand / 'final_dataset_metadata.csv').exists()
        if has_splits and (has_metadata or any((cand / 'labels' / 'train').glob('*.txt'))):
            return cand
    for cand in candidates:
        if all((cand / 'images' / s).is_dir() and (cand / 'labels' / s).is_dir()
               for s in ('train', 'val', 'test')):
            return cand
    raise FileNotFoundError('Could not locate a dataset with images/{train,val,test} and labels/{train,val,test}. Checked: ' + ', '.join(map(str, candidates)))

FINAL_DIR = resolve_dataset_root()
DATASET_ROOT = FINAL_DIR
IMAGES_ROOT = DATASET_ROOT / 'images'
LABELS_ROOT = DATASET_ROOT / 'labels'

EXPERIMENT_ROOT = PROJECT_ROOT / 'experiments' / EXPERIMENT_NAME
RUN_ROOT = EXPERIMENT_ROOT / EXPERIMENT_NAME
MODEL_ROOT = PROJECT_ROOT / 'models' / 'detection'
RESULT_ROOT = PROJECT_ROOT / 'results' / 'detection'

for directory in [EXPERIMENT_ROOT, MODEL_ROOT, RESULT_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

print('Model:', MODEL_NAME)
print('Dataset:', DATASET_ROOT)


In [ ]:
# Verify split directories and report image/label counts.
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
splits = ['train', 'val', 'test']
for split in splits:
    img_dir = IMAGES_ROOT / split
    lbl_dir = LABELS_ROOT / split
    if not img_dir.is_dir():
        raise FileNotFoundError(f'Missing image directory: {img_dir}')
    if not lbl_dir.is_dir():
        raise FileNotFoundError(f'Missing label directory: {lbl_dir}')
    images = [p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in image_exts]
    labels = list(lbl_dir.glob('*.txt'))
    if split in ('train', 'val') and not images:
        raise ValueError(f'No images found in {img_dir}')
    print(f'{split:>5}: images={len(images):,} labels={len(labels):,}')

print('Dataset structure verified. Empty label files are allowed for images with no objects.')


In [ ]:
import yaml

# Reuse the project class configuration when available so Detector 1 and Detector 2
# use the same class order. Otherwise use the same documented BDD-style fallback.
class_names = None
for cfg_name in ['detection.yaml', 'dataset.yaml']:
    cfg_path = PROJECT_ROOT / 'configs' / cfg_name
    if cfg_path.exists():
        try:
            cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8')) or {}
            candidate = cfg.get('names') or cfg.get('classes')
            if isinstance(candidate, dict):
                def class_key(value):
                    text = str(value)
                    return (0, int(text)) if text.isdigit() else (1, text)
                class_names = [candidate[k] for k in sorted(candidate, key=class_key)]
            elif isinstance(candidate, list):
                class_names = candidate
            if class_names:
                print(f'Loaded classes from {cfg_path}: {class_names}')
                break
        except Exception as e:
            print(f'Could not parse {cfg_path}: {e}')

if not class_names:
    class_names = ['pedestrian', 'rider', 'car', 'truck', 'bus', 'train', 'motorcycle', 'bicycle']
    print('Using fallback classes:', class_names)

if len(set(map(str, class_names))) != len(class_names):
    raise ValueError(f'Class names must be unique: {class_names}')
NUM_CLASSES = len(class_names)
print('Number of classes:', NUM_CLASSES)


In [ ]:
# Validate YOLO label syntax, finite normalized coordinates, and class IDs before training.
import math
label_rows = []
bad_rows = []
label_files_without_images = []

for split in splits:
    image_stems = {p.stem for p in (IMAGES_ROOT / split).iterdir()
                   if p.is_file() and p.suffix.lower() in image_exts}
    for label_path in sorted((LABELS_ROOT / split).glob('*.txt')):
        if label_path.stem not in image_stems:
            label_files_without_images.append((split, label_path.name))
        try:
            text = label_path.read_text(encoding='utf-8').strip()
            if not text:
                continue
            for line_no, line in enumerate(text.splitlines(), start=1):
                parts = line.split()
                if len(parts) != 5:
                    bad_rows.append((split, label_path.name, line_no, 'expected 5 fields'))
                    continue
                try:
                    cls_value = float(parts[0])
                    coords = [float(x) for x in parts[1:]]
                except ValueError:
                    bad_rows.append((split, label_path.name, line_no, 'non-numeric class or coordinate'))
                    continue
                if not math.isfinite(cls_value) or not cls_value.is_integer():
                    bad_rows.append((split, label_path.name, line_no, 'class ID must be a finite integer'))
                    continue
                cls = int(cls_value)
                if not (0 <= cls < NUM_CLASSES):
                    bad_rows.append((split, label_path.name, line_no, f'class {cls} out of range [0, {NUM_CLASSES - 1}]'))
                if any(not math.isfinite(x) for x in coords):
                    bad_rows.append((split, label_path.name, line_no, 'coordinates must be finite'))
                elif any((x < 0 or x > 1) for x in coords):
                    bad_rows.append((split, label_path.name, line_no, 'coordinate outside [0,1]'))
                label_rows.append((split, cls))
        except Exception as e:
            bad_rows.append((split, label_path.name, 0, str(e)))

print('Label rows inspected:', len(label_rows))
print('Invalid label records:', len(bad_rows))
if label_files_without_images:
    print('Warning: label files with no matching image:', len(label_files_without_images))
    print(pd.DataFrame(label_files_without_images, columns=['split', 'label_file']).head(10))
if bad_rows:
    print(pd.DataFrame(bad_rows, columns=['split', 'file', 'line', 'reason']).head(20))
    raise ValueError('Fix invalid YOLO labels before training.')


In [ ]:
# Create an Ultralytics dataset YAML.
dataset_yaml = EXPERIMENT_ROOT / 'detector_2_dataset.yaml'
yaml_data = {
    'path': str(DATASET_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': NUM_CLASSES,
    'names': {i: name for i, name in enumerate(class_names)},
}
dataset_yaml.write_text(yaml.safe_dump(yaml_data, sort_keys=False), encoding='utf-8')
print(dataset_yaml)
print(dataset_yaml.read_text(encoding='utf-8'))


## 2. Train Detector 2

The test split is not used for model selection. Ultralytics selects the best checkpoint from validation performance.

In [ ]:
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else 'cpu'
print('Training device:', device)

model = YOLO(MODEL_NAME)

train_results = model.train(
    data=str(dataset_yaml),
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    epochs=EPOCHS,
    patience=PATIENCE,
    workers=WORKERS,
    seed=SEED,
    optimizer='auto',
    project=str(EXPERIMENT_ROOT),
    name=EXPERIMENT_NAME,
    exist_ok=True,
    device=device,
    pretrained=True,
    plots=True,
    verbose=True,
)

print('Training complete.')


In [ ]:
# Locate and copy best/last checkpoints into the project's model directory.
best_src = RUN_ROOT / 'weights' / 'best.pt'
last_src = RUN_ROOT / 'weights' / 'last.pt'

if not best_src.exists():
    raise FileNotFoundError(f'Best checkpoint not found: {best_src}')

best_dst = MODEL_ROOT / 'detector_2_best.pt'
last_dst = MODEL_ROOT / 'detector_2_last.pt'
shutil.copy2(best_src, best_dst)
if last_src.exists():
    shutil.copy2(last_src, last_dst)

print('Best checkpoint:', best_dst)
print('Last checkpoint:', last_dst if last_dst.exists() else 'not found')


## 3. Validation evaluation

Run validation evaluation using the copied best checkpoint. This is the split used for model selection.

In [ ]:
best_model = YOLO(str(best_dst))
val_metrics = best_model.val(
    data=str(dataset_yaml),
    split='val',
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    device=device,
    project=str(EXPERIMENT_ROOT),
    name='detector_2_val',
    exist_ok=True,
    plots=True,
    verbose=True,
)
print('Validation evaluation complete.')


## 4. Final test evaluation

The test split is evaluated only after training/model selection.

In [ ]:
test_metrics = best_model.val(
    data=str(dataset_yaml),
    split='test',
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    device=device,
    project=str(EXPERIMENT_ROOT),
    name='detector_2_test',
    exist_ok=True,
    plots=True,
    verbose=True,
)
print('Test evaluation complete.')


In [ ]:
# Extract common Ultralytics detection metrics into a compact report.
def safe_metric(obj, path, default=None):
    cur = obj
    for key in path:
        try:
            cur = getattr(cur, key)
        except Exception:
            try:
                cur = cur[key]
            except Exception:
                return default
    try:
        return float(cur)
    except Exception:
        return default

def summarize_metrics(metrics):
    return {
        'map50': safe_metric(metrics, ['box', 'map50']),
        'map50_95': safe_metric(metrics, ['box', 'map']),
        'precision': safe_metric(metrics, ['box', 'mp']),
        'recall': safe_metric(metrics, ['box', 'mr']),
    }

val_summary = summarize_metrics(val_metrics)
test_summary = summarize_metrics(test_metrics)

print('Validation:', val_summary)
print('Test:', test_summary)


In [ ]:
# Save per-class test metrics when Ultralytics exposes them.
class_metrics_rows = []
box = getattr(test_metrics, 'box', None)
if box is not None:
    maps = getattr(box, 'maps', None)
    p = getattr(box, 'p', None)
    r = getattr(box, 'r', None)
    ap50 = getattr(box, 'ap50', None)
    ap = getattr(box, 'ap', None)

    for i, name in enumerate(class_names):
        row = {'class_id': i, 'class_name': name}
        for key, arr in [('precision', p), ('recall', r), ('map50', ap50), ('map50_95', maps), ('ap', ap)]:
            try:
                row[key] = float(arr[i]) if arr is not None and len(arr) > i else None
            except Exception:
                row[key] = None
        class_metrics_rows.append(row)

class_metrics_path = RESULT_ROOT / 'detector_2_test_class_metrics.csv'
pd.DataFrame(class_metrics_rows).to_csv(class_metrics_path, index=False)
print('Saved:', class_metrics_path)


In [ ]:
summary = {
    'experiment': EXPERIMENT_NAME,
    'model': MODEL_NAME,
    'project_root': str(PROJECT_ROOT),
    'dataset_yaml': str(dataset_yaml),
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'patience': PATIENCE,
    'workers': WORKERS,
    'seed': SEED,
    'classes': class_names,
    'best_checkpoint': str(best_dst),
    'last_checkpoint': str(last_dst) if last_dst.exists() else None,
    'validation_metrics': val_summary,
    'test_metrics': test_summary,
}

summary_path = RESULT_ROOT / 'detector_2_final_report.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

training_summary_path = RESULT_ROOT / 'detector_2_training_summary.json'
training_summary_path.write_text(json.dumps({
    'experiment': EXPERIMENT_NAME,
    'model': MODEL_NAME,
    'dataset_yaml': str(dataset_yaml),
    'best_checkpoint': str(best_dst),
    'validation_metrics': val_summary,
}, indent=2), encoding='utf-8')

print('Saved:', summary_path)
print('Saved:', training_summary_path)


## Expected outputs

- `experiments/detector_2/detector_2_dataset.yaml`
- `experiments/detector_2/detector_2/weights/best.pt`
- `models/detection/detector_2_best.pt`
- `models/detection/detector_2_last.pt`
- `results/detection/detector_2_test_class_metrics.csv`
- `results/detection/detector_2_training_summary.json`
- `results/detection/detector_2_final_report.json`

**Next:** Detector 3 can use the same protocol so the four detector experiments remain directly comparable.